In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("USE CATALOG workspace")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

def add_sk(df, nome_sk, ordem):
    """Surrogate key determinística via row_number(), ordenada por colunas estáveis.
    BIGINT conforme o enunciado. Determinística = reprocessar gera as mesmas chaves."""
    w = Window.orderBy(*ordem)
    return df.withColumn(nome_sk, F.row_number().over(w).cast("bigint"))

def salvar(df, nome):
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(f"gold.{nome}"))
    print(nome, "->", spark.table(f"gold.{nome}").count(), "linhas")

In [0]:
m = spark.table("silver.tb_info_filmes")

# Uma linha por filme (a Silver já garante unicidade por id_filme).
dim_movies = (add_sk(m, "sk_movie_id", ["id_filme"])
    .select(
        "sk_movie_id",
        F.col("id_filme").cast("string").alias("id_filme"),   # chave natural de origem
        F.col("titulo").cast("string"),
        F.col("data_lancamento").cast("date"),
        F.col("ano_lancamento").cast("int"),
        F.col("duracao_minutos").cast("int"),
        F.col("idioma_original").cast("string"),
        F.col("status_filme").cast("string"),
        F.col("sinopse").cast("string")))
salvar(dim_movies, "dim_movies")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_movies -> 97611 linhas


In [0]:
# ---------- dim_genres: catálogo único e deduplicado ----------
gen = spark.table("silver.tb_generos").select(F.col("genero").alias("nome_genero")).distinct()
dim_genres = add_sk(gen, "sk_genre_id", ["nome_genero"]).select("sk_genre_id", "nome_genero")
salvar(dim_genres, "dim_genres")

# ---------- dim_people: só pessoas físicas (Ator, Diretor, Roteirista) ----------
# Grão = (nome, tipo): a mesma pessoa que atua e dirige aparece em duas linhas, uma por papel.
pe = spark.table("silver.tb_pessoas_empresas")
pess = (pe.filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
          .select(F.col("nome_entidade").alias("nome_pessoa"),
                  F.col("tipo_entidade").alias("tipo_pessoa"))
          .distinct())
dim_people = add_sk(pess, "sk_person_id", ["nome_pessoa", "tipo_pessoa"]) \
               .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
salvar(dim_people, "dim_people")

# ---------- dim_companies: catálogo único de produtoras ----------
emp = (pe.filter(F.col("tipo_entidade") == "Produtora")
         .select(F.col("nome_entidade").alias("nome_produtora")).distinct())
dim_companies = add_sk(emp, "sk_company_id", ["nome_produtora"]) \
                  .select("sk_company_id", "nome_produtora")
salvar(dim_companies, "dim_companies")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_genres -> 19 linhas
dim_people -> 418605 linhas
dim_companies -> 45757 linhas


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
dim_movies = spark.table("gold.dim_movies")
fin = spark.table("silver.tb_financeiro_filmes")
met = spark.table("silver.tb_metricas_engajamento")

# Grão: 1 registro por filme. Só filmes lançados (status = 'Lançado'), como pede o enunciado.
# Os joins são LEFT a partir da dimensão e as duas Silver são únicas por id_filme,
# então o grão não se duplica.
lancados = dim_movies.filter(F.col("status_filme") == "Lançado").select("sk_movie_id", "id_filme")

fact = (lancados
    .join(fin, "id_filme", "left")
    .join(met, "id_filme", "left")
    .select(
        F.col("sk_movie_id").cast("bigint"),
        F.col("orcamento_usd").cast("decimal(18,2)"),
        F.col("receita_usd").cast("decimal(18,2)"),
        F.col("lucro_usd").cast("decimal(18,2)"),
        F.col("orcamento_brl").cast("decimal(18,2)"),
        F.col("receita_brl").cast("decimal(18,2)"),
        F.col("lucro_brl").cast("decimal(18,2)"),
        F.col("popularidade").cast("double"),
        F.col("nota_media_tmdb").cast("double"),
        F.col("qtd_votos_tmdb").cast("int"),
        F.col("nota_media_imdb").cast("double"),
        F.col("qtd_votos_imdb").cast("int")))
salvar(fact, "fact_movies_performance")

fact_movies_performance -> 96261 linhas


In [0]:
dim_movies = spark.table("gold.dim_movies")

# Avaliações individuais -> métrica resumida por filme.
# F.avg ignora NULL (notas fora de 0..10 já viraram NULL na Silver).
resumo = (spark.table("silver.tb_avaliacoes_usuarios")
    .groupBy("id_filme")
    .agg(F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
         F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios")))

# Inner join: só entram avaliações de filmes que existem em dim_movies (integridade da FK).
r = resumo.join(dim_movies.select("sk_movie_id", "id_filme"), "id_filme")
dim_reviews = add_sk(r, "sk_review_id", ["sk_movie_id"]) \
    .select("sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios")
salvar(dim_reviews, "dim_reviews")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_reviews -> 27226 linhas


In [0]:
mv = spark.table("gold.dim_movies").select("sk_movie_id", "id_filme")
dg = spark.table("gold.dim_genres")
dp = spark.table("gold.dim_people")
dc = spark.table("gold.dim_companies")
pe = spark.table("silver.tb_pessoas_empresas")

# Pontes resolvem o N:N (filme x gênero/pessoa/produtora) sem duplicar a fato.
# distinct() garante que a mesma relação não apareça duas vezes.

# --- filme x gênero ---
b_genre = (spark.table("silver.tb_generos")
           .join(mv, "id_filme")
           .join(dg, F.col("genero") == F.col("nome_genero"))
           .select("sk_movie_id", "sk_genre_id").distinct())
salvar(b_genre, "bridge_movie_genre")

# --- filme x pessoa (Ator, Diretor, Roteirista) ---
# Renomeia antes do join para evitar colunas ambíguas.
pe_pessoas = (pe.filter(F.col("tipo_entidade") != "Produtora")
                .select("id_filme",
                        F.col("nome_entidade").alias("nome_p"),
                        F.col("tipo_entidade").alias("tipo_p")))
b_person = (pe_pessoas.join(mv, "id_filme")
            .join(dp, (F.col("nome_p") == dp.nome_pessoa) & (F.col("tipo_p") == dp.tipo_pessoa))
            .select("sk_movie_id", "sk_person_id").distinct())
salvar(b_person, "bridge_movie_person")

# --- filme x produtora ---
pe_prod = (pe.filter(F.col("tipo_entidade") == "Produtora")
             .select("id_filme", F.col("nome_entidade").alias("nome_prod")))
b_comp = (pe_prod.join(mv, "id_filme")
          .join(dc, F.col("nome_prod") == dc.nome_produtora)
          .select("sk_movie_id", "sk_company_id").distinct())
salvar(b_comp, "bridge_movie_company")

bridge_movie_genre -> 139859 linhas
bridge_movie_person -> 758097 linhas
bridge_movie_company -> 118814 linhas


In [0]:
tabelas = ["dim_movies", "dim_genres", "dim_people", "dim_companies", "dim_reviews",
           "fact_movies_performance", "bridge_movie_genre", "bridge_movie_person", "bridge_movie_company"]
for t in tabelas:
    print(f"{t}: {spark.table('gold.' + t).count()}")

# Grão da fato: 1 linha por filme (deve dar 0)
print("sk_movie_id duplicado na fato:", spark.sql("""
  SELECT COUNT(*) c FROM (SELECT sk_movie_id FROM gold.fact_movies_performance
  GROUP BY sk_movie_id HAVING COUNT(*) > 1)""").first()["c"])

# Integridade referencial: FKs órfãs (todas devem dar 0)
checks = {
  "fato -> dim_movies": "SELECT COUNT(*) c FROM gold.fact_movies_performance f LEFT JOIN gold.dim_movies m ON f.sk_movie_id=m.sk_movie_id WHERE m.sk_movie_id IS NULL",
  "reviews -> dim_movies": "SELECT COUNT(*) c FROM gold.dim_reviews r LEFT JOIN gold.dim_movies m ON r.sk_movie_id=m.sk_movie_id WHERE m.sk_movie_id IS NULL",
  "bridge genre": "SELECT COUNT(*) c FROM gold.bridge_movie_genre b LEFT JOIN gold.dim_genres g ON b.sk_genre_id=g.sk_genre_id WHERE g.sk_genre_id IS NULL",
  "bridge person": "SELECT COUNT(*) c FROM gold.bridge_movie_person b LEFT JOIN gold.dim_people p ON b.sk_person_id=p.sk_person_id WHERE p.sk_person_id IS NULL",
  "bridge company": "SELECT COUNT(*) c FROM gold.bridge_movie_company b LEFT JOIN gold.dim_companies c2 ON b.sk_company_id=c2.sk_company_id WHERE c2.sk_company_id IS NULL",
}
for nome, q in checks.items():
    print("órfãos", nome, ":", spark.sql(q).first()["c"])

# Tipos conforme o enunciado
spark.table("gold.fact_movies_performance").printSchema()
display(spark.table("gold.dim_movies").limit(5))

dim_movies: 97611
dim_genres: 19
dim_people: 418605
dim_companies: 45757
dim_reviews: 27226
fact_movies_performance: 96261
bridge_movie_genre: 139859
bridge_movie_person: 758097
bridge_movie_company: 118814
sk_movie_id duplicado na fato: 0
órfãos fato -> dim_movies : 0
órfãos reviews -> dim_movies : 0
órfãos bridge genre : 0
órfãos bridge person : 0
órfãos bridge company : 0
root
 |-- sk_movie_id: long (nullable = true)
 |-- orcamento_usd: decimal(18,2) (nullable = true)
 |-- receita_usd: decimal(18,2) (nullable = true)
 |-- lucro_usd: decimal(18,2) (nullable = true)
 |-- orcamento_brl: decimal(18,2) (nullable = true)
 |-- receita_brl: decimal(18,2) (nullable = true)
 |-- lucro_brl: decimal(18,2) (nullable = true)
 |-- popularidade: double (nullable = true)
 |-- nota_media_tmdb: double (nullable = true)
 |-- qtd_votos_tmdb: integer (nullable = true)
 |-- nota_media_imdb: double (nullable = true)
 |-- qtd_votos_imdb: integer (nullable = true)



sk_movie_id,id_filme,titulo,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse
1,1000004,Purple Beatz,2022-07-07,2022,86,en,Lançado,"Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry."
2,1000005,Aisha Brown: The First Black Woman Ever,2020-02-14,2020,42,en,Lançado,"No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs."
3,1000007,KYLE BROWNRIGG: INTRODUCING LYLE,2022-05-27,2022,36,en,Lançado,"Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle."
4,1000011,Worth Your Weight in Gold,2022-07-14,2022,26,pt,Lançado,"Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness."
5,1000014,On va manquer !,2018-05-15,2018,0,fr,Lançado,null
